In [223]:
#!pip install catboost optuna lightgbm xgboost

In [224]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.metrics import mean_squared_error as MSE
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split,KFold
import optuna
from xgboost import XGBRegressor
import xgboost as xgb
import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import KFold

In [225]:
from sklearn.preprocessing import SplineTransformer


class SplineTargetEncoder:
    def __init__(self, degree=3, n_knots=5):
        self.degree = degree
        self.n_knots = n_knots
        self.spline_transformer = None

    def fit(self, X, y, continuous_feature):
        self.spline_transformer = SplineTransformer(degree=self.degree, n_knots=self.n_knots, include_bias=False)
        self.spline_transformer.fit(X[[continuous_feature]])  
    def transform(self, X, continuous_feature):
        spline_features = self.spline_transformer.transform(X[[continuous_feature]])

        spline_df = pd.DataFrame(spline_features, columns=[f"{continuous_feature}_spline_{i}" for i in range(spline_features.shape[1])])
        return spline_df

    def fit_transform(self, X, y, continuous_feature):
        self.fit(X, y, continuous_feature)
        return self.transform(X, continuous_feature)


spline = SplineTargetEncoder(degree=3, n_knots=5)


In [226]:
import warnings
warnings.simplefilter('ignore')

In [227]:
SEED=95
TRIALS=70
BINS=5

In [228]:
df=pd.read_csv('train.csv')
df.head()

,id,Brand,Material,Size,Compartments,Laptop Compartment,Waterproof,Style,Color,Weight Capacity (kg),Price
0,0,Jansport,Leather,Medium,7.0,Yes,No,Tote,Black,11.611723,112.15875
1,1,Jansport,Canvas,Small,10.0,Yes,Yes,Messenger,Green,27.078537,68.88056
2,2,Under Armour,Leather,Small,2.0,Yes,No,Messenger,Red,16.643760,39.17320
3,3,Nike,Nylon,Small,8.0,Yes,No,Messenger,Green,12.937220,80.60793
4,4,Adidas,Canvas,Medium,1.0,Yes,Yes,Messenger,Green,17.749338,86.02312


In [107]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Dense
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[['Weight Capacity (kg)']])  

X_train, X_test, y_train, y_test = train_test_split(X_scaled, df.Price, test_size=0.2, random_state=42)

model = keras.Sequential([
    Dense(256, activation="relu", input_shape=(1,)),
    Dense(256, activation="relu"),
    Dense(1)  
])


model.compile(optimizer="adam", loss="mse", metrics=[tf.keras.metrics.RootMeanSquaredError()])

model.fit(X_train, y_train, epochs=20, batch_size=64, verbose=1, validation_split=0.1)

rmse = model.evaluate(X_test, y_test, verbose=0)[1]
print("RMSE:", rmse)

Epoch 1/20
3375/3375 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - loss: 2082.8372 - root_mean_squared_error: 44.8836 - val_loss: 1521.0101 - val_root_mean_squared_error: 39.0001
Epoch 2/20
3375/3375 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - loss: 1529.8844 - root_mean_squared_error: 39.1135 - val_loss: 1520.8872 - val_root_mean_squared_error: 38.9986
Epoch 3/20
3375/3375 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - loss: 1528.5934 - root_mean_squared_error: 39.0971 - val_loss: 1519.6716 - val_root_mean_squared_error: 38.9830
Epoch 4/20
3375/3375 ━━━━━━━━━━━━━━━━━━━━ 15s 5ms/step - loss: 1530.3365 - root_mean_squared_error: 39.1195 - val_loss: 1521.5941 - val_root_mean_squared_error: 39.0076
Epoch 5/20
3375/3375 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - loss: 1529.8467 - root_mean_squared_error: 39.1132 - val_loss: 1523.0922 - val_root_mean_squared_error: 39.0268
Epoch 6/20
3375/3375 ━━━━━━━━━━━━━━━━━━━━ 15s 5ms/step - loss: 1530.4427 - root_mean_squared_error: 39.1206 - val_loss: 1522.4875 - val_root_mean_squared_e

In [229]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 11 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    300000 non-null  int64  
 1   Brand                 290295 non-null  object 
 2   Material              291653 non-null  object 
 3   Size                  293405 non-null  object 
 4   Compartments          300000 non-null  float64
 5   Laptop Compartment    292556 non-null  object 
 6   Waterproof            292950 non-null  object 
 7   Style                 292030 non-null  object 
 8   Color                 290050 non-null  object 
 9   Weight Capacity (kg)  299862 non-null  float64
 10  Price                 300000 non-null  float64
dtypes: float64(3), int64(1), object(7)
memory usage: 25.2+ MB


In [230]:
df["Weight Capacity (kg)"].describe()

count    299862.000000
mean         18.029994
std           6.966914
min           5.000000
25%          12.097867
50%          18.068614
75%          24.002375
max          30.000000
Name: Weight Capacity (kg), dtype: float64

In [231]:
categorical_cols = ["Brand", "Material", "Size", "Laptop Compartment", "Waterproof", "Style", "Color", "Compartments"]
numerical_cols = ["Weight Capacity (kg)"]
df['Compartments'] = df['Compartments'].astype('int').astype('object')
#df[categorical_cols] = df[categorical_cols].fillna('Unknown')
#df[numerical_cols] = df[numerical_cols].fillna(df[numerical_cols].median())

def impute_and_encode(train_df, test_df, categorical_cols, numerical_cols):
    train_df, test_df = train_df.copy(), test_df.copy()

    # Numerical Imputation: Fill NaNs with train median
    median_values = train_df[numerical_cols].median()
    train_df[numerical_cols] = train_df[numerical_cols].fillna(median_values)
    test_df[numerical_cols] = test_df[numerical_cols].fillna(median_values)

    for col in categorical_cols:
        mode_value = train_df[col].mode()[0]
        train_df[col] = train_df[col].fillna(mode_value)
        test_df[col] = test_df[col].fillna(mode_value)

    # Final check to ensure no NaNs remain
    assert train_df.isna().sum().sum() == 0, "Train data still contains NaN values!"
    assert test_df.isna().sum().sum() == 0, "Test data still contains NaN values!"


    #bins = pd.cut(df['Weight Capacity (kg)'], bins=BINS, retbins=True)[1]
    #train_df['Weight'] = pd.cut(train_df['Weight Capacity (kg)'], bins=bins, labels=False, include_lowest=True)
    #test_df['Weight'] = pd.cut(test_df['Weight Capacity (kg)'], bins=bins, labels=False, include_lowest=True)

    train_df['Weight'] =train_df['Weight Capacity (kg)'].astype('str')
    test_df['Weight'] = test_df['Weight Capacity (kg)'].astype('str')

    categorical_cols.append("Weight")
    return train_df, test_df

df,_ = impute_and_encode(df, df, categorical_cols, numerical_cols)


In [232]:
cat_cols = categorical_cols
X_,y_=df[['Brand','Material','Size','Compartments','Laptop Compartment','Waterproof','Style','Color','Weight Capacity (kg)','Weight']],df.Price

In [233]:
'''
def cat_objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'random_strength': trial.suggest_float('random_strength', 0.1, 10),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'verbose': False,
    }
    
    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    rmse_scores = []

    for train_idx, val_idx in kf.split(X_):
        X_train, X_val = X_.iloc[train_idx], X_.iloc[val_idx]
        y_train, y_val = y_.iloc[train_idx], y_.iloc[val_idx]

        train_pool = Pool(X_train, label=y_train, cat_features=cat_cols)
        val_pool = Pool(X_val, label=y_val, cat_features=cat_cols)

        model = CatBoostRegressor(**params)
        model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=50, verbose=False)
        
        y_pred = model.predict(val_pool)
        rmse = np.sqrt(MSE(y_val, y_pred))
        rmse_scores.append(rmse)

    return np.mean(rmse_scores)  

study_cat = optuna.create_study(study_name='CatBoost_KFold', direction='minimize')
optuna.logging.set_verbosity(optuna.logging.WARNING)
study_cat.optimize(cat_objective, n_trials=TRIALS, show_progress_bar=True)

print('Best hyperparameters:', study_cat.best_params)
best_params = study_cat.best_params


final_model = CatBoostRegressor(**best_params, verbose=False)'''

"\ndef cat_objective(trial):\n    params = {\n        'iterations': trial.suggest_int('iterations', 100, 1000),\n        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),\n        'depth': trial.suggest_int('depth', 4, 10),\n        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),\n        'border_count': trial.suggest_int('border_count', 32, 255),\n        'random_strength': trial.suggest_float('random_strength', 0.1, 10),\n        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),\n        'verbose': False,\n    }\n    \n    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)\n    rmse_scores = []\n\n    for train_idx, val_idx in kf.split(X_):\n        X_train, X_val = X_.iloc[train_idx], X_.iloc[val_idx]\n        y_train, y_val = y_.iloc[train_idx], y_.iloc[val_idx]\n\n        train_pool = Pool(X_train, label=y_train, cat_features=cat_cols)\n        val_pool = Pool(X_val, label=y_val, cat_features=cat_cols)\n\n        model = Ca

In [234]:
best_params ={'iterations': 606, 'learning_rate': 0.10464287163726789, 'depth': 4, 'l2_leaf_reg': 5.39240446712957, 'border_count': 225, 'random_strength': 5.766329375236162, 'bagging_temperature': 0.30902928577805355}
final_model = CatBoostRegressor(**best_params, verbose=False)

In [235]:
final_pool = Pool(X_, label=y_, cat_features=cat_cols)
final_model.fit(final_pool)
print('Test RMSE for Cat is '+str(np.sqrt(MSE(y_,final_model.predict(X_)))))

Test RMSE for Cat is 37.66406522324392


In [236]:
df_=pd.read_csv('test.csv')
categorical_cols = ["Brand", "Material", "Size", "Laptop Compartment", "Waterproof", "Style", "Color", "Compartments"]
numerical_cols = ["Weight Capacity (kg)"]
df_['Compartments'] = df_['Compartments'].astype('int').astype('object')
#df_[categorical_cols] = df_[categorical_cols].fillna('Unknown')
#df_[numerical_cols] = df_[numerical_cols].fillna(df[numerical_cols].median())
_,df_ = impute_and_encode(df, df_, categorical_cols, numerical_cols)
X=df_[['Brand','Material','Size','Compartments','Laptop Compartment','Waterproof','Style','Color','Weight Capacity (kg)','Weight']]


submission=pd.read_csv('sample_submission.csv')
submission['Price']=final_model.predict(X)
submission.set_index('id').to_csv('submissionCatx.csv')

In [237]:
df.isna().sum()

id                      0
Brand                   0
Material                0
Size                    0
Compartments            0
Laptop Compartment      0
Waterproof              0
Style                   0
Color                   0
Weight Capacity (kg)    0
Price                   0
Weight                  0
dtype: int64

In [238]:
df.head()

,id,Brand,Material,Size,Compartments,Laptop Compartment,Waterproof,Style,Color,Weight Capacity (kg),Price,Weight
0,0,Jansport,Leather,Medium,7,Yes,No,Tote,Black,11.611723,112.15875,11.611722805222309
1,1,Jansport,Canvas,Small,10,Yes,Yes,Messenger,Green,27.078537,68.88056,27.07853658053123
2,2,Under Armour,Leather,Small,2,Yes,No,Messenger,Red,16.643760,39.17320,16.643759949103497
3,3,Nike,Nylon,Small,8,Yes,No,Messenger,Green,12.937220,80.60793,12.937220306632067
4,4,Adidas,Canvas,Medium,1,Yes,Yes,Messenger,Green,17.749338,86.02312,17.749338465908988


In [239]:
# RUN FROM HERE

SEED=95
TRIALS=70
BINS=5

In [240]:
categorical_cols = ["Brand", "Material", "Size", "Laptop Compartment", "Waterproof", "Style", "Color", "Compartments"]
numerical_cols = ["Weight Capacity (kg)"]
df['Compartments'] = df['Compartments'].astype('int').astype('object')

def impute_and_encode(train_df, test_df, categorical_cols, numerical_cols):
    train_df, test_df = train_df.copy(), test_df.copy()
    median_values = train_df[numerical_cols].median()
    train_df[numerical_cols] = train_df[numerical_cols].fillna(median_values)
    test_df[numerical_cols] = test_df[numerical_cols].fillna(median_values)

    for col in categorical_cols:
        mode_value = train_df[col].mode()[0]
        train_df[col] = train_df[col].fillna(mode_value)
        test_df[col] = test_df[col].fillna(mode_value)

    assert train_df.isna().sum().sum() == 0, "Train data still contains NaN values!"
    assert test_df.isna().sum().sum() == 0, "Test data still contains NaN values!"
    bins = pd.cut(df['Weight Capacity (kg)'], bins=BINS, retbins=True)[1]
    train_df['Weight'] = pd.cut(train_df['Weight Capacity (kg)'], bins=bins, labels=False, include_lowest=True)
    test_df['Weight'] = pd.cut(test_df['Weight Capacity (kg)'], bins=bins, labels=False, include_lowest=True)

    categorical_cols.append("Weight")
    return train_df, test_df
df,_ = impute_and_encode(df, df, categorical_cols, numerical_cols)

In [241]:
import category_encoders as ce

target_encoder = ce.TargetEncoder(cols=categorical_cols)


df[categorical_cols] = target_encoder.fit_transform(df[categorical_cols], df["Price"])
df = pd.concat([df,spline.fit_transform(df, df['Price'], "Weight Capacity (kg)")],axis=1)

In [242]:
df.head()

,id,Brand,Material,Size,Compartments,Laptop Compartment,Waterproof,Style,Color,Weight Capacity (kg),Price,Weight,Weight Capacity (kg)_spline_0,Weight Capacity (kg)_spline_1,Weight Capacity (kg)_spline_2,Weight Capacity (kg)_spline_3,Weight Capacity (kg)_spline_4,Weight Capacity (kg)_spline_5
0,0,81.791276,80.437883,81.180993,81.440569,81.361493,81.572050,81.374865,80.513439,11.611723,112.15875,81.561777,0.0,0.139371,0.663414,0.197182,0.000032,0.000000
1,1,81.791276,82.106511,81.424674,81.685283,81.361493,81.260802,81.451328,82.381308,27.078537,68.88056,81.879968,0.0,0.000000,0.000000,0.017022,0.458565,0.499238
2,2,81.976311,80.437883,81.424674,81.171776,81.361493,81.572050,81.451328,81.011644,16.643760,39.17320,82.046565,0.0,0.000429,0.243265,0.649184,0.107123,0.000000
3,3,81.319209,81.024760,81.424674,81.906747,81.361493,81.572050,81.451328,82.381308,12.937220,80.60793,81.561777,0.0,0.064848,0.603627,0.328246,0.003279,0.000000
4,4,80.664673,82.106511,81.180993,81.263845,81.361493,81.260802,81.451328,82.381308,17.749338,86.02312,82.046565,0.0,0.000000,0.147505,0.665107,0.187378,0.000011


In [243]:
df.isna().sum()

id                               0
Brand                            0
Material                         0
Size                             0
Compartments                     0
Laptop Compartment               0
Waterproof                       0
Style                            0
Color                            0
Weight Capacity (kg)             0
Price                            0
Weight                           0
Weight Capacity (kg)_spline_0    0
Weight Capacity (kg)_spline_1    0
Weight Capacity (kg)_spline_2    0
Weight Capacity (kg)_spline_3    0
Weight Capacity (kg)_spline_4    0
Weight Capacity (kg)_spline_5    0
dtype: int64

In [244]:
X,y=df.drop(columns=['Price']),df.Price
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

In [245]:
X.isna().sum().sum()

0

In [246]:
scaler=StandardScaler()
X__=scaler.fit_transform(X)
X=scaler.fit_transform(X)

In [247]:
'''
def ridge_objective(trial):
    alpha = trial.suggest_float("alpha", 1e-5, 10.0, log=True)  # Regularization strength
    fit_intercept = trial.suggest_categorical("fit_intercept", [True, False])

    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    rmse_scores = []

    for train_idx, val_idx in kf.split(X__):
        X_train, X_val = X__[train_idx], X__[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)

        model = Ridge(alpha=alpha, fit_intercept=fit_intercept)
        model.fit(X_train_scaled, y_train)

        y_pred = model.predict(X_val_scaled)
        rmse = np.sqrt(MSE(y_val, y_pred))
        rmse_scores.append(rmse)

    return np.mean(rmse_scores)

study_ridge = optuna.create_study(study_name="Ridge_Backpack", direction="minimize")
optuna.logging.set_verbosity(optuna.logging.WARNING)
study_ridge.optimize(ridge_objective, n_trials=50, show_progress_bar=True)

print("Best Ridge Trial:")
print(f" Value: {study_ridge.best_trial.value:.4f}")
print(" Params: ")
print(study_ridge.best_params)

ridge_model = Ridge(**study_ridge.best_params)'''

'\ndef ridge_objective(trial):\n    alpha = trial.suggest_float("alpha", 1e-5, 10.0, log=True)  # Regularization strength\n    fit_intercept = trial.suggest_categorical("fit_intercept", [True, False])\n\n    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)\n    rmse_scores = []\n\n    for train_idx, val_idx in kf.split(X__):\n        X_train, X_val = X__[train_idx], X__[val_idx]\n        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]\n\n        scaler = StandardScaler()\n        X_train_scaled = scaler.fit_transform(X_train)\n        X_val_scaled = scaler.transform(X_val)\n\n        model = Ridge(alpha=alpha, fit_intercept=fit_intercept)\n        model.fit(X_train_scaled, y_train)\n\n        y_pred = model.predict(X_val_scaled)\n        rmse = np.sqrt(MSE(y_val, y_pred))\n        rmse_scores.append(rmse)\n\n    return np.mean(rmse_scores)\n\nstudy_ridge = optuna.create_study(study_name="Ridge_Backpack", direction="minimize")\noptuna.logging.set_verbosity(optuna.logging.WARN

In [248]:
ridge_model = Ridge(**{'alpha': 9.777033857638354, 'fit_intercept': True})

In [249]:
ridge_model.fit(X,y)

Ridge(alpha=9.777033857638354)

In [250]:
'''
def lasso_objective(trial):
    alpha = trial.suggest_float("alpha", 1e-5, 10.0, log=True)  # L1 penalty
    fit_intercept = trial.suggest_categorical("fit_intercept", [True, False])

    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    rmse_scores = []

    for train_idx, val_idx in kf.split(X__):
        X_train, X_val = X__[train_idx], X__[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)

        model = Lasso(alpha=alpha, fit_intercept=fit_intercept)
        model.fit(X_train_scaled, y_train)

        y_pred = model.predict(X_val_scaled)
        rmse = np.sqrt(MSE(y_val, y_pred))
        rmse_scores.append(rmse)

    return np.mean(rmse_scores)

study_lasso = optuna.create_study(study_name="Lasso_Backpack", direction="minimize")
optuna.logging.set_verbosity(optuna.logging.WARNING)
study_lasso.optimize(lasso_objective, n_trials=TRIALS, show_progress_bar=True)

print("Best Lasso Trial:")
print(f" Value: {study_lasso.best_trial.value:.4f}")
print(" Params: ")
print(study_lasso.best_params)

lasso_model = Lasso(**study_lasso.best_params)'''

'\ndef lasso_objective(trial):\n    alpha = trial.suggest_float("alpha", 1e-5, 10.0, log=True)  # L1 penalty\n    fit_intercept = trial.suggest_categorical("fit_intercept", [True, False])\n\n    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)\n    rmse_scores = []\n\n    for train_idx, val_idx in kf.split(X__):\n        X_train, X_val = X__[train_idx], X__[val_idx]\n        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]\n\n        scaler = StandardScaler()\n        X_train_scaled = scaler.fit_transform(X_train)\n        X_val_scaled = scaler.transform(X_val)\n\n        model = Lasso(alpha=alpha, fit_intercept=fit_intercept)\n        model.fit(X_train_scaled, y_train)\n\n        y_pred = model.predict(X_val_scaled)\n        rmse = np.sqrt(MSE(y_val, y_pred))\n        rmse_scores.append(rmse)\n\n    return np.mean(rmse_scores)\n\nstudy_lasso = optuna.create_study(study_name="Lasso_Backpack", direction="minimize")\noptuna.logging.set_verbosity(optuna.logging.WARNING)\nstudy_l

In [251]:
lasso_model = Lasso(**{'alpha': 1.0029293434060904e-05, 'fit_intercept': True})

In [252]:
lasso_model.fit(X,y)

Lasso(alpha=1.0029293434060904e-05)

In [253]:
'''
from tensorflow import keras
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout, LeakyReLU
import tensorflow as tf
from tensorflow.keras.losses import MeanSquaredError


def root_mean_squared_error(y_true, y_pred):
    return tf.keras.backend.sqrt(MeanSquaredError()(y_true, y_pred))


def Dl_objective(trial):
    dropout_rate = trial.suggest_float("dropout", 0.1, 0.5)
    leaky_relu_alpha = trial.suggest_float("leaky_relu_alpha", 0.01, 0.3)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])

    def create_model():
        model = keras.Sequential([
            Dense(256), BatchNormalization(), LeakyReLU(alpha=leaky_relu_alpha), Dropout(dropout_rate),
            Dense(256), BatchNormalization(), LeakyReLU(alpha=leaky_relu_alpha), Dropout(dropout_rate),
            Dense(1, activation="linear")  
        ])

        model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
                      loss="mse",
                      metrics=[root_mean_squared_error])
        return model

    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    fold_rmse_scores = []

    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        model = create_model()
        model.fit(X_train, y_train, validation_data=(X_val, y_val),
                  epochs=10, batch_size=batch_size, verbose=1)

        _, rmse = model.evaluate(X_val, y_val, verbose=0)
        fold_rmse_scores.append(rmse)

    return np.mean(fold_rmse_scores)

study = optuna.create_study(study_name="Dl_Backpack", direction="minimize")
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(Dl_objective, n_trials=40, show_progress_bar=True)


print("Best Hyperparameters:", study.best_params)'''


'\nfrom tensorflow import keras\nfrom tensorflow.keras.layers import Dense, BatchNormalization, Dropout, LeakyReLU\nimport tensorflow as tf\nfrom tensorflow.keras.losses import MeanSquaredError\n\n\ndef root_mean_squared_error(y_true, y_pred):\n    return tf.keras.backend.sqrt(MeanSquaredError()(y_true, y_pred))\n\n\ndef Dl_objective(trial):\n    dropout_rate = trial.suggest_float("dropout", 0.1, 0.5)\n    leaky_relu_alpha = trial.suggest_float("leaky_relu_alpha", 0.01, 0.3)\n    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)\n    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])\n\n    def create_model():\n        model = keras.Sequential([\n            Dense(256), BatchNormalization(), LeakyReLU(alpha=leaky_relu_alpha), Dropout(dropout_rate),\n            Dense(256), BatchNormalization(), LeakyReLU(alpha=leaky_relu_alpha), Dropout(dropout_rate),\n            Dense(1, activation="linear")  \n        ])\n\n        model.compile(optimizer=ke

In [257]:
best_params_dl={'dropout': 0.48114544194204734, 'leaky_relu_alpha': 0.10849916932430939, 'learning_rate': 0.0011485940818786846, 'batch_size': 64}

def create_model(leaky_relu_alpha=0.10849916932430939, dropout_rate=0.48114544194204734, learning_rate=0.0011485940818786846):
        model = keras.Sequential([
            Dense(256), BatchNormalization(), LeakyReLU(alpha=leaky_relu_alpha), Dropout(dropout_rate),
            Dense(256), BatchNormalization(), LeakyReLU(alpha=leaky_relu_alpha), Dropout(dropout_rate),
            Dense(1, activation="linear")  
        ])

        model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate),loss="mse",metrics=[root_mean_squared_error])
        return model

Dl_model=create_model()

Dl_model.fit(X,y,epochs=32,batch_size=best_params_dl['batch_size'],verbose=1)

Epoch 1/32
4688/4688 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - loss: 2542.1240 - root_mean_squared_error: 47.9428
Epoch 2/32
4688/4688 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 1577.8291 - root_mean_squared_error: 39.6511
Epoch 3/32
4688/4688 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 1567.7721 - root_mean_squared_error: 39.5273
Epoch 4/32
4688/4688 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 1561.8892 - root_mean_squared_error: 39.4500
Epoch 5/32
4688/4688 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 1561.7334 - root_mean_squared_error: 39.4528
Epoch 6/32
4688/4688 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 1560.5374 - root_mean_squared_error: 39.4355
Epoch 7/32
4688/4688 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 1551.5320 - root_mean_squared_error: 39.3215
Epoch 8/32
4688/4688 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 1548.7534 - root_mean_squared_error: 39.2866
Epoch 9/32
4688/4688 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 1552.1066 - root_mean_squared_error: 39.3325
Epoch 10/32
4688/4688 ━━━━━━━━━━━━━━━

In [258]:
'''
def lgbm_objective(trial):
    lgbm_params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "subsample": trial.suggest_float("subsample", 0.3, 0.9),
        "min_child_samples": trial.suggest_int("min_child_samples", 60, 100),
        "max_depth": trial.suggest_int("max_depth", 4, 25),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.001, 0.1),
        "lambda_l2": trial.suggest_float("lambda_l2", 0.001, 0.1),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
        "random_state": SEED
    }

    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    rmse_scores = []

    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = lgb.LGBMRegressor(**lgbm_params, verbose=-1)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_val)
        rmse = np.sqrt(MSE(y_val, y_pred))
        rmse_scores.append(rmse)

    return np.mean(rmse_scores)

study_LGBM = optuna.create_study(study_name="LGBM_Backpack", direction="minimize")
optuna.logging.set_verbosity(optuna.logging.WARNING)
study_LGBM.optimize(lgbm_objective, n_trials=TRIALS, show_progress_bar=True)

print("Best Trial:")
print(f" Value: {study_LGBM.best_trial.value:.4f}")
print(" Params: ")
print(study_LGBM.best_params)

recognizer = lgb.LGBMRegressor(**study_LGBM.best_params, random_state=SEED, verbose=-1)'''

Best trial: 63. Best value: 39.0142: 100%|██████████| 70/70 [14:07<00:00, 12.11s/it]

Best Trial:
 Value: 39.0142
 Params: 
{'n_estimators': 182, 'subsample': 0.4712428828676451, 'min_child_samples': 64, 'max_depth': 7, 'learning_rate': 0.01908958455886573, 'lambda_l1': 0.09473571370255374, 'lambda_l2': 0.08569139463025688, 'colsample_bytree': 0.8213658221911804}


In [263]:
Lgbm_best_params={'n_estimators': 182, 'subsample': 0.4712428828676451, 'min_child_samples': 64, 'max_depth': 7, 'learning_rate': 0.01908958455886573, 'lambda_l1': 0.09473571370255374, 'lambda_l2': 0.08569139463025688, 'colsample_bytree': 0.8213658221911804}
recognizer = lgb.LGBMRegressor(**Lgbm_best_params,random_state=SEED,verbose=-1)

In [264]:
recognizer.fit(X,y)
print('Test RMSE for LGBM is '+str(np.sqrt(MSE(y,recognizer.predict(X)))))

Test RMSE for LGBM is 38.935185208638


In [265]:
X,y=df.drop(columns=['Price']),df.Price

In [267]:
df_=pd.read_csv('test.csv')
df_['Compartments'] = df_['Compartments'].astype('int').astype('object')
categorical_cols = ["Brand", "Material", "Size", "Laptop Compartment", "Waterproof", "Style", "Color", "Compartments"]
numerical_cols = ["Weight Capacity (kg)"]
_,df_ = impute_and_encode(df, df_, categorical_cols, numerical_cols)

'''
categorical_cols = ["Brand", "Material", "Size", "Laptop Compartment", "Waterproof", "Style", "Color", "Compartments"]
numerical_cols = ["Weight Capacity (kg)"]
df=pd.read_csv('train.csv')
df[categorical_cols] = df[categorical_cols].fillna('Unknown')'''


X_=df_
#print(categorical_cols)
X_[categorical_cols]=target_encoder.transform(X[categorical_cols])
X_[numerical_cols] = X[numerical_cols].fillna(X[numerical_cols].median())
X_ = pd.concat([X_,spline.transform(X_,"Weight Capacity (kg)")],axis=1)
X_=scaler.fit_transform(X_)


submission=pd.read_csv('sample_submission.csv')
submission['Price']=ridge_model.predict(X_)
print(submission.Price.isna().sum())
submission.set_index('id').to_csv('submissionRidge.csv')

0


In [268]:
df_=pd.read_csv('test.csv')
df_['Compartments'] = df_['Compartments'].astype('int').astype('object')
categorical_cols = ["Brand", "Material", "Size", "Laptop Compartment", "Waterproof", "Style", "Color", "Compartments"]
numerical_cols = ["Weight Capacity (kg)"]
_,df_ = impute_and_encode(df, df_, categorical_cols, numerical_cols)

'''
categorical_cols = ["Brand", "Material", "Size", "Laptop Compartment", "Waterproof", "Style", "Color", "Compartments"]
numerical_cols = ["Weight Capacity (kg)"]
df=pd.read_csv('train.csv')
df[categorical_cols] = df[categorical_cols].fillna('Unknown')'''


X_=df_
#print(categorical_cols)
X_[categorical_cols]=target_encoder.transform(X[categorical_cols])
X_[numerical_cols] = X[numerical_cols].fillna(X[numerical_cols].median())
X_ = pd.concat([X_,spline.transform(X_,"Weight Capacity (kg)")],axis=1)
X_=scaler.fit_transform(X_)


submission=pd.read_csv('sample_submission.csv')
submission['Price']=ridge_model.predict(X_)
print(submission.Price.isna().sum())
submission.set_index('id').to_csv('submissionLasso.csv')

0


In [269]:
df_=pd.read_csv('test.csv')
df_['Compartments'] = df_['Compartments'].astype('int').astype('object')
categorical_cols = ["Brand", "Material", "Size", "Laptop Compartment", "Waterproof", "Style", "Color", "Compartments"]
numerical_cols = ["Weight Capacity (kg)"]
_,df_ = impute_and_encode(df, df_, categorical_cols, numerical_cols)

'''
categorical_cols = ["Brand", "Material", "Size", "Laptop Compartment", "Waterproof", "Style", "Color", "Compartments"]
numerical_cols = ["Weight Capacity (kg)"]
df=pd.read_csv('train.csv')
df[categorical_cols] = df[categorical_cols].fillna('Unknown')'''


X_=df_
#print(categorical_cols)
X_[categorical_cols]=target_encoder.transform(X[categorical_cols])
X_[numerical_cols] = X[numerical_cols].fillna(X[numerical_cols].median())
X_ = pd.concat([X_,spline.transform(X_,"Weight Capacity (kg)")],axis=1)
X_=scaler.fit_transform(X_)


submission=pd.read_csv('sample_submission.csv')
submission['Price']=ridge_model.predict(X_)
print(submission.Price.isna().sum())
submission.set_index('id').to_csv('submissionDl.csv')

0


In [270]:
df_=pd.read_csv('test.csv')
df_['Compartments'] = df_['Compartments'].astype('int').astype('object')
categorical_cols = ["Brand", "Material", "Size", "Laptop Compartment", "Waterproof", "Style", "Color", "Compartments"]
numerical_cols = ["Weight Capacity (kg)"]
_,df_ = impute_and_encode(df, df_, categorical_cols, numerical_cols)

'''
categorical_cols = ["Brand", "Material", "Size", "Laptop Compartment", "Waterproof", "Style", "Color", "Compartments"]
numerical_cols = ["Weight Capacity (kg)"]
df=pd.read_csv('train.csv')
df[categorical_cols] = df[categorical_cols].fillna('Unknown')'''


X_=df_
#print(categorical_cols)
X_[categorical_cols]=target_encoder.transform(X[categorical_cols])
X_[numerical_cols] = X[numerical_cols].fillna(X[numerical_cols].median())
X_ = pd.concat([X_,spline.transform(X_,"Weight Capacity (kg)")],axis=1)
X_=scaler.fit_transform(X_)


submission=pd.read_csv('sample_submission.csv')
submission['Price']=ridge_model.predict(X_)
print(submission.Price.isna().sum())
submission.set_index('id').to_csv('submissionLGBM.csv')

0


In [271]:
X=scaler.fit_transform(X)

In [272]:
def rf_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 1000, log=True),
        "max_depth": trial.suggest_int("max_depth", 4, 128),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "random_state": SEED,
        "n_jobs": -1
    }

    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    rmse_scores = []

    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = RandomForestRegressor(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_val)
        rmse = np.sqrt(MSE(y_val, y_pred))
        rmse_scores.append(rmse)

    return np.mean(rmse_scores)

study_Rf = optuna.create_study(study_name="Rf_Backpack", direction="minimize")
optuna.logging.set_verbosity(optuna.logging.WARNING)
study_Rf.optimize(rf_objective, n_trials=TRIALS, show_progress_bar=True)

print("Best Trial:")
print(f" Value: {study_Rf.best_trial.value:.4f}")
print(" Params: ")
print(study_Rf.best_params)

recognizer = RandomForestRegressor(**study_Rf.best_params, random_state=SEED, n_jobs=-1)

  0%|          | 0/70 [06:47<?, ?it/s]


[W 2025-02-04 17:51:58,283] Trial 0 failed with parameters: {'n_estimators': 527, 'max_depth': 40, 'min_samples_split': 6, 'min_samples_leaf': 4} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/diptyajit.das/.local/lib/python3.12/site-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_10236/1318718397.py", line 19, in rf_objective
    model.fit(X_train, y_train)
  File "/home/diptyajit.das/.local/lib/python3.12/site-packages/sklearn/base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/diptyajit.das/.local/lib/python3.12/site-packages/sklearn/ensemble/_forest.py", line 489, in fit
    trees = Parallel(
            ^^^^^^^^^
  File "/home/diptyajit.das/.local/lib/python3.12/site-packages/sklearn/utils/parallel.py", line 74, in __call__
    retu

KeyboardInterrupt: 

In [96]:
'''
Rf_best_params= {'n_estimators': 188, 'max_depth': 13, 'min_samples_split': 8, 'min_samples_leaf': 8}
recognizer = RandomForestRegressor(**Rf_best_params,random_state=SEED)'''

In [ ]:
recognizer.fit(X,y)
print('Test RMSE for Rf is '+str(np.sqrt(MSE(y,recognizer.predict(X)))))

In [98]:
X,y=df[['Brand','Material','Size','Compartments','Laptop Compartment','Waterproof','Style','Color','Weight Capacity (kg)','Weight']],df.Price

In [ ]:
df_=pd.read_csv('test.csv')
df_['Compartments'] = df_['Compartments'].astype('int').astype('object')
categorical_cols = ["Brand", "Material", "Size", "Laptop Compartment", "Waterproof", "Style", "Color", "Compartments"]
numerical_cols = ["Weight Capacity (kg)"]
_,df_ = impute_and_encode(df, df_, categorical_cols, numerical_cols)

'''
categorical_cols = ["Brand", "Material", "Size", "Laptop Compartment", "Waterproof", "Style", "Color", "Compartments"]
numerical_cols = ["Weight Capacity (kg)"]
df=pd.read_csv('train.csv')
df[categorical_cols] = df[categorical_cols].fillna('Unknown')'''


X_=df_
#print(categorical_cols)
X_[categorical_cols]=target_encoder.transform(X[categorical_cols])
X_[numerical_cols] = X[numerical_cols].fillna(X[numerical_cols].median())
X_ = pd.concat([X_,spline.transform(X_,"Weight Capacity (kg)")],axis=1)
X_=scaler.fit_transform(X_)


submission=pd.read_csv('sample_submission.csv')
submission['Price']=ridge_model.predict(X_)
print(submission.Price.isna().sum())
submission.set_index('id').to_csv('submissionRf.csv')

In [ ]:
'''
import numpy as np
import pandas as pd


cat=pd.read_csv('submissionCatx.csv').Price
rf=pd.read_csv('submissionRf.csv').Price
lgb=pd.read_csv('submissionLGBM.csv').Price
la=pd.read_csv('submissionLasso.csv').Price
ri=pd.read_csv('submissionRidge.csv').Price

submission=pd.read_csv('sample_submission.csv')
submission['Price']= .2*cat+.2*rf+.2*lgb+.2*la+.2*ri
print(submission['Price'].isna().sum())
print(len(submission.Price))
submission.set_index('id').to_csv('submissionensemble.csv')'''

In [103]:
import numpy as np
import pandas as pd


cat=pd.read_csv('submissionCat.csv').Price
cat2=pd.read_csv('submissionCat2.csv').Price
bias=pd.read_csv('bias.csv').Price

submission=pd.read_csv('sample_submission.csv')
submission['Price']= .7*cat + .3*cat2
submission.set_index('id').to_csv('submission.csv')

In [100]:
X=scaler.fit_transform(X)

In [ ]:
def xgb_objective(trial):
    param = {
        'booster': trial.suggest_categorical('booster', ['gbtree', 'dart']),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_weight': trial.suggest_float('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'lambda': trial.suggest_float('lambda', 1e-3, 10.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-3, 10.0, log=True),
        'random_state': SEED
    }

    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    rmse_scores = []

    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        model = XGBRegressor(**param, verbosity=0)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_val)
        rmse = np.sqrt(MSE(y_val, y_pred))
        rmse_scores.append(rmse)

    return np.mean(rmse_scores)

study_XGB = optuna.create_study(study_name='XGB_Backpack', direction='minimize')
optuna.logging.set_verbosity(optuna.logging.WARNING)
study_XGB.optimize(xgb_objective, n_trials=TRIALS, show_progress_bar=True)

print("Best Trial:")
print(f" Value: {study_XGB.best_trial.value:.4f}")
print(" Params: ")
print(study_XGB.best_params)

recognizer = XGBRegressor(**study_XGB.best_params, random_state=SEED, verbosity=0)

In [179]:
'''
Xgb_best_params={'booster': 'gbtree', 'learning_rate': 0.04337229505530977, 'n_estimators': 110, 'max_depth': 4, 'min_child_weight': 8.083204545400763, 'gamma': 1.4811492909447175, 'subsample': 0.9437523780997022, 'colsample_bytree': 0.862432875642509, 'lambda': 0.8922410191742034, 'alpha': 4.957212919990214}
recognizer = XGBRegressor(**Xgb_best_params,random_state=SEED,verbose=-1)'''

In [ ]:
recognizer.fit(X, y)
print('Test RMSE for XGB is '+str(np.sqrt(MSE(y,recognizer.predict(X)))))

In [ ]:
X,y=df[['Brand','Material','Size','Compartments','Laptop Compartment','Waterproof','Style','Color','Weight Capacity (kg)','Weight']],df.Price

In [181]:
df_=pd.read_csv('test.csv')
df_['Compartments'] = df_['Compartments'].astype('int').astype('object')
categorical_cols = ["Brand", "Material", "Size", "Laptop Compartment", "Waterproof", "Style", "Color", "Compartments"]
numerical_cols = ["Weight Capacity (kg)"]
_,df_ = impute_and_encode(df, df_, categorical_cols, numerical_cols)

'''
categorical_cols = ["Brand", "Material", "Size", "Laptop Compartment", "Waterproof", "Style", "Color", "Compartments"]
numerical_cols = ["Weight Capacity (kg)"]
df=pd.read_csv('train.csv')
df[categorical_cols] = df[categorical_cols].fillna('Unknown')'''


X_=df_.drop(columns=['Price'])
#print(categorical_cols)
X_[categorical_cols]=target_encoder.transform(X[categorical_cols])
X_[numerical_cols] = X[numerical_cols].fillna(X[numerical_cols].median())
X_ = pd.concat([X_,spline.transform(X_,"Weight Capacity (kg)")],axis=1)
X_=scaler.fit_transform(X_)


submission=pd.read_csv('sample_submission.csv')
submission['Price']=ridge_model.predict(X_)
print(submission.Price.isna().sum())
submission.set_index('id').to_csv('submissionXGB.csv')

In [2]:
import numpy as np
import pandas as pd

a,b,c,d=pd.read_csv('best.csv').Price,pd.read_csv('2nd.csv').Price,pd.read_csv('submissionCat.csv').Price,pd.read_csv('submissionCat2.csv').Price

con=(a+b+pd.read_csv('submissionensemble.csv').Price)/3

sub=(c+d)/2

unc=(.25*pd.read_csv('submissionDl.csv').Price+.25*pd.read_csv('submissionLGBM.csv').Price+.25*pd.read_csv('submissionLasso.csv').Price+.25*pd.read_csv('submissionRidge.csv').Price)

submission=pd.read_csv('sample_submission.csv')
submission['Price']= .7*con+.2*sub+.1*unc
submission.set_index('id').to_csv('submission.csv')

print(submission.Price.isna().sum().sum())

0
